# Examen bimestral

**Introducción a los Sistemas de Información**

Prof. Iván Carrera

Ingeniería en Ciencia de Datos e Inteligencia Artificial · Escuela Politécnica Nacional

**Calificación total:** 10 puntos.

## Caso de estudio · Papelería universitaria

Una papelería universitaria registra las compras de sus sucursales Central, Norte y Sur en un sistema de punto de venta. Cada compra genera un ticket que puede incluir varios productos. La persona encargada de la papelería necesita resumir estos registros para conocer qué se vendió, cómo se pagaron las compras y cómo variaron las ventas entre sucursales y meses.

Elabore los reportes solicitados a partir de los datos.

## Archivos de trabajo

| Archivo      | Contenido |
|--------------|---|
| `productos.csv` | Una fila por producto. Incluye su código, nombre, categoría y precio de lista. |
| `clientes.csv` | Una fila por cliente. Incluye su identificador, nombre, ciudad y tipo de cliente. |
| `ventas.csv`   | Una fila por producto dentro de un ticket. Incluye la fecha, sucursal, cliente, cantidad, precio unitario y método de pago. |

El campo `id_ticket` puede repetirse en varias filas de `ventas.csv`. Para relacionar los archivos, use `id_producto` en ventas y productos, y `ventas.id_cliente` con `clientes.id`. Un campo vacío en `metodo_pago` indica que ese dato quedó sin registrar.

## Preguntas

1. **Líneas, tickets y unidades.** ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

2. **Tickets por sucursal.** ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

3. **Métodos de pago.** ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

4. **Precios de lista por categoría.** ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

5. **Unidades por sucursal y categoría.** ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de `cantidad` en `ventas.csv`.

6. **Importe de ventas por sucursal.** ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como `cantidad × precio_unitario`.

7. **Tamaño de las compras.** ¿Cuántos tickets corresponden a compras de **hasta 7 unidades**, **de 8 a 14 unidades** y **de 15 o más unidades**? Sume primero las cantidades de todas las líneas de cada ticket.

8. **Ventas por mes.** ¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

9. **Producto más vendido por sucursal.** ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

10. **Compras por tipo de cliente.** Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

## Entrega y evaluación

Resuelva los reportes solicitados en un mismo Jupyter Notebook, identificados con el número de cada pregunta. Presente los reportes con encabezados claros y los importes en dólares con dos decimales. Use conteos, agrupaciones y cálculos descriptivos.

Entregue el link al Jupyter Notebook cargado en un repositorio Git en la Tarea del aula virtual.

**Cada pregunta vale 1 punto, para un total de 10 puntos.** En cada respuesta se evaluará que el código produzca el resultado solicitado a partir de los CSV y que se presenten todos los elementos pedidos en la pregunta.

Buena suerte.

In [111]:
import pandas as pd

dfventas = pd.read_csv('ventas.csv')
dfproductos = pd.read_csv('productos_.csv')
dfclientes = pd.read_csv('clientes.csv')

### Ejercicio 1
Líneas, tickets y unidades. ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

In [112]:
total_lineas = len(dfventas)
total_tickets_distintos = dfventas['id_ticket'].nunique()
unidades_vendidas = dfventas['cantidad'].sum()

tabla1 = pd.DataFrame({
    'Métrica': ['Líneas de venta', 'Tickets distintos', 'Unidades vendidas'],
    'Valores': [total_lineas, total_tickets_distintos, unidades_vendidas]
})
display(tabla1)


,Métrica,Valores
0,Líneas de venta,2100
1,Tickets distintos,300
2,Unidades vendidas,3880


### Ejercicio 2
Tickets por sucursal. ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

In [113]:
tickets_por_sucursal = dfventas.groupby('sucursal')['id_ticket'].nunique()

tabla3 = tickets_por_sucursal.rename_axis('Sucursal').reset_index(
    name='Numero de Tickets'
)

sucursal_max = tabla3.loc[tabla3['Numero de Tickets'].idxmax(), 'Sucursal']
max_tickets = tabla3['Numero de Tickets'].max()

tabla3.loc[len(tabla3)] = ['Total', dfventas['id_ticket'].nunique()]
display(tabla3)

print(f"Sucursal con más tickets: {sucursal_max} ({max_tickets} tickets)")

,Sucursal,Numero de Tickets
0,Central,146
1,Norte,83
2,Sur,71
3,Total,300


Sucursal con más tickets: Central (146 tickets)


### Ejercicio 3
Métodos de pago. ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

In [114]:
dfventas['metodo_pago_reporte'] = dfventas['metodo_pago'].fillna('Sin registrar')

tickets_por_metodo = (
    dfventas.groupby('metodo_pago_reporte')['id_ticket']
    .nunique()
    .sort_values(ascending=False)
)

tabla3 = tickets_por_metodo.rename_axis('Metodo de Pago').reset_index(
    name='Numero de Tickets'
)

metodo_max = tabla3.loc[tabla3['Numero de Tickets'].idxmax(), 'Metodo de Pago']
max_tickets = tabla3['Numero de Tickets'].max()

tabla3.loc[len(tabla3)] = ['Total', dfventas['id_ticket'].nunique()]

display(tabla3)

,Metodo de Pago,Numero de Tickets
0,Efectivo,144
1,Tarjeta,84
2,Transferencia,48
3,Sin registrar,24
4,Total,300


### Ejercicio 4
Precios de lista por categoría. ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

In [115]:
precio = dfproductos.groupby(dfproductos['categoria'])['precio_lista'].agg(['mean', 'min', 'max'])

tabla4 = pd.DataFrame({
    'Categoria': precio.index,
    'Precio promedio USD': precio['mean'].round(2).values,
    'Precio mínimo USD': precio['min'].values,
    'Precio máximo USD': precio['max'].values,
})
display(tabla4)

,Categoria,Precio promedio USD,Precio mínimo USD,Precio máximo USD
0,Accesorios,1.62,0.40,22.90
1,Archivo,1.67,0.35,3.95
2,Escritura,1.35,0.40,3.75
3,Papel y cuadernos,1.67,0.28,5.75


### Ejercicio 5
Unidades por sucursal y categoría. ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de cantidad en ventas.csv.

In [116]:
ventas_con_categoria = dfventas.merge(
    dfproductos[['id_producto', 'categoria']],
    on='id_producto',
    how='left'
)

unidades_sucursal_categoria = pd.crosstab(
    ventas_con_categoria['sucursal'],
    ventas_con_categoria['categoria'],
    values=ventas_con_categoria['cantidad'],
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).fillna(0).astype(int)

tabla5 = unidades_sucursal_categoria.rename_axis('Sucursal').reset_index()
display(tabla5)

print("Total de la tabla:", unidades_sucursal_categoria.loc['Total', 'Total'])
print("Suma de cantidad en ventas:", dfventas['cantidad'].sum())

print("¿Son iguales?", unidades_sucursal_categoria.loc['Total', 'Total'] == dfventas['cantidad'].sum())

categoria,Sucursal,Accesorios,Archivo,Escritura,Papel y cuadernos,Total
0,Central,453,481,437,524,1895
1,Norte,256,264,306,259,1085
2,Sur,191,223,238,248,900
3,Total,900,968,981,1031,3880


Total de la tabla: 3880
Suma de cantidad en ventas: 3880
¿Son iguales? True


### Ejercicio 6
Importe de ventas por sucursal. ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como cantidad × precio_unitario.

In [117]:
ventas_sucursal = dfventas.copy()
ventas_sucursal['importe_linea'] = (
    ventas_sucursal['cantidad'] * ventas_sucursal['precio_unitario']
)

tabla6 = (
    ventas_sucursal.groupby('sucursal', as_index=False)['importe_linea']
    .sum()
    .rename(columns={
        'sucursal': 'Sucursal',
        'importe_linea': 'Total de Importe USD'
    })
)
tabla6['Total de Importe USD'] = tabla6['Total de Importe USD'].round(2)

sucursal = tabla6.loc[tabla6['Total de Importe USD'].idxmax(), 'Sucursal']
total_importe_sucursal = tabla6.loc[
    tabla6['Sucursal'] == sucursal, 'Total de Importe USD'
]
display(tabla6)

total_ventas_general = dfventas['cantidad'] * dfventas['precio_unitario']
print(f"Importe total(USD) en todas las sucursales: {total_ventas_general.sum().round(2)}")

,Sucursal,Total de Importe USD
0,Central,3025.04
1,Norte,1635.35
2,Sur,1443.24


Importe total(USD) en todas las sucursales: 6103.63


### Ejercicio 7
Tamaño de las compras. ¿Cuántos tickets corresponden a compras de hasta 7 unidades, de 8 a 14 unidades y de 15 o más unidades? Sume primero las cantidades de todas las líneas de cada ticket.

In [118]:
filas = []
for ticket, id_producto in dfventas.groupby('id_ticket')['id_producto'].nunique().items():
    unidades_por_ticket = dfventas.groupby('id_ticket')['cantidad'].sum()

    conteos = {
        'Hasta 7 unidades': (unidades_por_ticket <= 7).sum(),
        'De 8 a 14 unidades': unidades_por_ticket.between(8, 14).sum(),
        '15 o más unidades': (unidades_por_ticket >= 15).sum()
    }

    tabla7 = pd.DataFrame(
        list(conteos.items()),
        columns=['Tamaño de compra', 'Número de tickets']
    )
    tabla7.loc[len(tabla7)] = ['Total', unidades_por_ticket.size]

display(tabla7)

,Tamaño de compra,Número de tickets
0,Hasta 7 unidades,39
1,De 8 a 14 unidades,153
2,15 o más unidades,108
3,Total,300


### Ejercicio 8
Ventas por mes. ¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

In [119]:
tickets = dfventas.copy()
tickets['fecha'] = pd.to_datetime(tickets['fecha'], errors='coerce')
tickets['importe_linea'] = tickets['cantidad'] * tickets['precio_unitario']
tickets['mes'] = tickets['fecha'].dt.to_period('M')

tabla8 = (
    tickets.groupby('mes')
    .agg(
            Numero_de_tickets=('id_ticket', 'nunique'),
            Importe_total_USD=('importe_linea', 'sum')
        )
    .reset_index()
)
tabla8['Importe_total_USD'] = tabla8['Importe_total_USD'].round(2)

display(tabla8)

mes_maximo = tabla8.loc[tabla8['Importe_total_USD'].idxmax()]
print(
    f"Mes con mayor importe: {mes_maximo['mes']} "
    f"(${mes_maximo['Importe_total_USD']:.2f})"
)

,mes,Numero_de_tickets,Importe_total_USD
0,2025-11,25,492.51
1,2025-12,25,645.54
2,2026-01,25,447.48
3,2026-02,25,466.36
4,2026-03,25,595.33
5,2026-04,25,505.43
6,2026-05,25,453.94
7,2026-06,25,500.54
8,2026-07,25,499.88
9,2026-08,25,478.79


Mes con mayor importe: 2025-12 ($645.54)


### Ejercicio 9
Producto más vendido por sucursal. ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

In [120]:
ventas_producto = (
    dfventas
    .merge(dfproductos[['id_producto', 'producto']], on='id_producto', how='left')
    .groupby(['sucursal', 'id_producto', 'producto'], as_index=False)['cantidad']
    .sum()
    .rename(columns={'cantidad': 'unidades_vendidas'})
)

max_por_sucursal = (
    ventas_producto.groupby('sucursal')['unidades_vendidas']
    .transform('max')
)

producto_mas_vendido_por_sucursal = (
    ventas_producto[ventas_producto['unidades_vendidas'] == max_por_sucursal]
    .sort_values(['sucursal', 'id_producto'])
    .reset_index(drop=True)
)

producto_mas_vendido_por_sucursal = producto_mas_vendido_por_sucursal.rename(
    columns={
        'sucursal': 'Sucursal',
        'id_producto': 'Código del producto',
        'producto': 'Nombre del producto',
        'unidades_vendidas': 'Unidades vendidas'
    }
)

display(producto_mas_vendido_por_sucursal)

,Sucursal,Código del producto,Nombre del producto,Unidades vendidas
0,Central,P399,Estuche escolar (gris),17
1,Norte,P141,Marcador de punta fina (naranja),17
2,Sur,P016,Cuaderno cuadriculado A5 (24 hojas),11
3,Sur,P304,Portadocumentos con cierre (rojo),11


### Ejercicio 10
Compras por tipo de cliente. Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

In [121]:
ventas_con_clientes = (
    dfventas
    .assign(importe_linea=lambda x: x['cantidad'] * x['precio_unitario'])
    .merge(
        dfclientes[['id', 'tipo_cliente']],
        left_on='id_cliente',
        right_on='id',
        how='left'
    )
    .drop(columns=['id'])
)

clientes_distintos = (
    ventas_con_clientes
    .groupby('tipo_cliente', as_index=False)
    .agg(
        Clientes_distintos=('id_cliente', 'nunique'),
        Tickets=('id_ticket', 'nunique'),
        Importe_total=('importe_linea', 'sum')
    )
    .rename(columns={
        'tipo_cliente': 'Tipo de cliente',
        'Clientes_distintos': 'Clientes distintos'
    })
)

clientes_distintos['Importe total USD'] = clientes_distintos['Importe_total'].round(2)
clientes_distintos = (
    clientes_distintos
    .set_index('Tipo de cliente')
    .reindex(['Docente', 'Estudiante', 'Personal administrativo'])
    .reset_index()
    [['Tipo de cliente', 'Clientes distintos', 'Tickets', 'Importe total USD']]
)

display(clientes_distintos)

,Tipo de cliente,Clientes distintos,Tickets,Importe total USD
0,Docente,16,46,899.74
1,Estudiante,72,226,4688.00
2,Personal administrativo,12,28,515.89
